# EditGuard — Visual Model Editing Workbench

A private Colab interface for **real BLIP-2 parameter edits**, generated before/after answers, patch export, and exact rollback. Choose the published **MEND editor with adapted labels** or **last-layer fine-tuning (15 steps, adapted baseline)**. Both are experimental; neither makes generated text a verified fact.

**Saved measured comparison:** the completed sealed T4 experiment contains 20 edits and 500 generated-answer records. FT matched all 60 edited targets but none of the 40 locality targets, breaking all five baseline-correct locality answers. MEND retained those five answers while matching fewer edit targets; RAG and prompt-only each matched 54/60 edited targets. These are results on one small nonrandom benchmark slice, not an arbitrary-edit guarantee. Read the [results and limitations](../docs/VLKEB_RESULTS.md). The notebook installs that saved, checksum-verified report for the dashboard; it does **not** rerun the benchmark.

The earlier [MEND development evidence](../reports/vlkeb/t4-context-vqa-1/run/smoke.json) and [FT development evidence](../reports/vlkeb/t4-ft-last-layer-1/run/smoke.json) remain separate diagnostics on one training-seen example.

The authors of [VLKEB](https://github.com/VLKEB/VLKEB) and [MEND](https://arxiv.org/abs/2110.11309) supply the research core and pretrained editor. Our work adds the compatibility adapter, experiment controls, transactional patch lifecycle and interface. The upstream revision is pinned below. This is **not** a claim that we trained MEND or reproduced every paper result. Published results must remain attributed upstream, separate from our measurements.

Use **Runtime → Change runtime type → GPU**. A T4 was measured in our development sessions; availability and memory can vary. No Google Drive mount, paid upgrade, public tunnel or additional web framework is used. OPT is subject to its noncommercial research license; model, editor, annotations and images have separate terms in the asset manifest. Read any displayed license terms yourself.

Run cells in order. Downloads are explicit, and later stages refuse to run after a failed setup stage. Keep only one full model process on the GPU.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import os
import platform
import shutil
import socket
import subprocess
import sys
import time
import urllib.request
import uuid
import zipfile

REPOSITORY_URL = "https://github.com/ShivamRaghav7/verity.git"
WRAPPER_COMMIT = "e0324c39e545e8222ed99ca6148bdebf9c790465"
UPSTREAM_COMMIT = "10951b7b3788928f578b73f07eac9e1eaa0316f3"
REPOSITORY = Path("/content/editguard-vlkeb")
ENVIRONMENT = REPOSITORY / "artifacts/vlkeb/environment"
PYTHON = ENVIRONMENT / "bin/python"
UPSTREAM = REPOSITORY / "tmp/vlkeb-upstream"
ASSETS = REPOSITORY / "artifacts/vlkeb/assets"
STORAGE = REPOSITORY / "artifacts/vlkeb/workbench"
EVALUATION = REPOSITORY / "artifacts/vlkeb/eval20/evaluation.json"
SAVED_EVALUATION_SHA256 = "db9e89cdfad68841de99a5e185aacd661fdf2afee07409a9dd774b400e2d433d"
PORT = 8003
STAGES = {}

def require_stage(name):
    if STAGES.get(name) is not True:
        raise RuntimeError(f"Complete the successful {name!r} stage first.")

def run_checked(command, timeout=1800):
    print("Running:", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=REPOSITORY,
                          text=True, check=True, timeout=timeout)

if platform.system() != "Linux" or platform.machine().lower() != "x86_64" or not Path("/content").is_dir():
    raise RuntimeError("Use a Linux x86_64 Colab runtime. This leaves the Windows environment unchanged.")
try:
    query = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version",
         "--format=csv,noheader,nounits"], text=True, timeout=30)
except (OSError, subprocess.SubprocessError) as exc:
    raise RuntimeError("No NVIDIA GPU is available. Choose Runtime → Change runtime type → GPU (T4 if offered), then rerun.") from exc
gpus = [{"name": row[0].strip(), "total_vram_mib": int(row[1]),
         "free_vram_mib": int(row[2]), "driver": row[3].strip()}
        for row in csv.reader(io.StringIO(query)) if row]
if not gpus:
    raise RuntimeError("No GPU is assigned. Enable a GPU runtime before continuing.")
inventory = {"utc": datetime.now(timezone.utc).isoformat(), "gpus": gpus,
             "disk_free_bytes": shutil.disk_usage("/content").free,
             "notice": "Inventory only; not an accuracy or guaranteed-fit claim."}
print(json.dumps(inventory, indent=2))
STAGES["hardware"] = True


## 1. Fetch the exact application revision

This creates an owned detached checkout at a fixed commit. A compatible existing checkout is reused only if its remote matches and its tracked files are clean. We never reset, force-checkout, delete user work, or change the Windows installation. Untracked files under `artifacts/` remain local.

This revision includes the completed measured evaluation and current dashboard. After checkout, the next cell verifies the saved report's SHA-256 and copies it into the dashboard's default report directory only when absent. An identical existing copy is reused; a different existing file is preserved and stops setup. This copies public saved evidence, with no model inference, tuning or new evaluation.


In [ ]:
require_stage("hardware")
STAGES["checkout"] = False
if not REPOSITORY.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY)],
                   check=True, text=True, timeout=300)
elif not (REPOSITORY / ".git").is_dir():
    raise RuntimeError("The target already exists without this Git checkout; choose a fresh REPOSITORY directory.")
remote = subprocess.check_output(["git", "-C", str(REPOSITORY), "remote", "get-url", "origin"],
                                 text=True, timeout=20).strip()
if remote.rstrip("/") != REPOSITORY_URL:
    raise RuntimeError("Existing checkout has a different remote; no changes were made.")
dirty = subprocess.check_output(["git", "-C", str(REPOSITORY), "status",
                                 "--porcelain", "--untracked-files=no"], text=True, timeout=20)
if dirty.strip():
    raise RuntimeError("Tracked files have local changes. Preserve them and choose a fresh directory; this notebook will not reset them.")
subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "origin", WRAPPER_COMMIT],
               check=True, text=True, timeout=300)
subprocess.run(["git", "-C", str(REPOSITORY), "checkout", "--detach", WRAPPER_COMMIT],
               check=True, text=True, timeout=60)
actual = subprocess.check_output(["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"],
                                 text=True, timeout=20).strip()
if actual != WRAPPER_COMMIT:
    raise RuntimeError("Wrapper revision did not match the declared pin.")
os.chdir(REPOSITORY)
inventory_path = REPOSITORY / "artifacts/vlkeb/workbench-runtime-inventory.json"
inventory_path.parent.mkdir(parents=True, exist_ok=True)
inventory_path.write_text(json.dumps({**inventory, "wrapper_commit": actual}, indent=2) + "\n")
def install_saved_evaluation():
    source = REPOSITORY / "reports/vlkeb/t4-eval20-1/artifacts/vlkeb/eval20/evaluation.json"
    saved_bytes = source.read_bytes()
    if hashlib.sha256(saved_bytes).hexdigest() != SAVED_EVALUATION_SHA256:
        raise RuntimeError("Saved public evaluation checksum differs from the declared receipt; no report was copied.")
    if EVALUATION.exists():
        if not EVALUATION.is_file() or hashlib.sha256(EVALUATION.read_bytes()).hexdigest() != SAVED_EVALUATION_SHA256:
            raise FileExistsError("The dashboard report already exists with different content. Preserve it and choose a fresh checkout; this notebook will not overwrite it.")
        action = "reused_identical_saved_report"
    else:
        EVALUATION.parent.mkdir(parents=True, exist_ok=True)
        with EVALUATION.open("xb") as stream:
            stream.write(saved_bytes)
        action = "copied_saved_report"
    print(json.dumps({"evaluation_report": str(EVALUATION), "sha256": SAVED_EVALUATION_SHA256,
                      "action": action, "provenance": "saved own-measured completed T4 evaluation",
                      "new_model_run_executed": False}, indent=2))

install_saved_evaluation()
STAGES["checkout"] = True
print("Pinned checkout:", actual)


## 2. Prepare the isolated runtime

The next cell explicitly installs the pinned research dependencies into a separate Python 3.11 environment and fetches VLKEB at its recorded commit. It does not replace Colab's notebook Python, download models, train an editor or start the server.

A prior fresh setup took about 52 seconds in one session. That is an observation, not a promise; downloads and cache state affect future runs. Model execution has its own memory and latency costs.


In [ ]:
require_stage("checkout")
STAGES["environment"] = False
run_checked([sys.executable, "scripts/bootstrap_vlkeb_colab.py",
             "--fetch-upstream", "--install"], timeout=1800)
check = (
    "import json, torch; "
    "from pathlib import Path; "
    "from ke_repro.bridge import _upstream_modules; "
    "_upstream_modules(Path('tmp/vlkeb-upstream')); "
    "assert torch.cuda.is_available(), 'CUDA is not available in the isolated runtime'; "
    "print(json.dumps({'cuda_available': True, 'gpu': torch.cuda.get_device_name(0), "
    "'selective_upstream_import': 'passed'}))"
)
run_checked([PYTHON, "-c", check], timeout=180)
STAGES["environment"] = True


## 3. Inspect and explicitly prepare the model assets

The plan cell below makes no download. The following **download cell** explicitly fetches the pinned files, validates their hashes and prepares only the fixed development training fixture. It reuses valid cached files and can resume interrupted downloads. Do not substitute newer weights if a pinned source is unavailable.

The selected files total **13,218,483,144 bytes** (about 12.31 GiB), excluding dependencies and saved patches. A prior download took roughly five minutes in one session; network speed varies. Keep extra disk space: one saved six-matrix patch is about 600 MiB. Download size is not VRAM use.


In [ ]:
require_stage("environment")
STAGES["asset_plan"] = False
manifest = json.loads((REPOSITORY / "configs/vlkeb-assets.json").read_text())
asset_bytes = sum(item["size_bytes"] for item in manifest["assets"])
print(json.dumps({"declared_asset_bytes": asset_bytes,
                  "free_disk_bytes": shutil.disk_usage(REPOSITORY).free,
                  "existing_assets_may_be_reused": ASSETS.is_dir()}, indent=2))
run_checked([PYTHON, "scripts/prepare_vlkeb_assets.py", "--root", ASSETS], timeout=60)
STAGES["asset_plan"] = True


In [ ]:
# EXPLICIT DOWNLOAD: run this cell to fetch the reviewed asset plan above.
require_stage("asset_plan")
STAGES["assets"] = False
run_checked([PYTHON, "scripts/prepare_vlkeb_assets.py", "--root", ASSETS, "--download"], timeout=3600)
# An offline integrity check also verifies the prepared fixture against its archive.
run_checked([PYTHON, "scripts/prepare_vlkeb_assets.py", "--root", ASSETS, "--verify"], timeout=600)
STAGES["assets"] = True
print("Pinned assets and development fixture verified. No editing or training has run.")


## 4. Start one private workbench process

Stop any separate smoke/evaluation model worker before this cell. The server validates the assets again, then loads its backend in a background thread. It binds only to `127.0.0.1:8003` and serializes model operations. The notebook must stay connected.

MEND uses the released learned transform with our context-aligned label adaptation. The optional fine-tuning baseline takes **15 AdamW steps at learning rate 0.0002** on layer 31's fc1/fc2 weights. Its FP32 master optimizer is a memory adaptation; it is not a faithful whole-model-FP32 reproduction. Targets supervise updates; targets are not inserted into generation prompts.

An existing process is never killed automatically. Each launch gets a new log. No web request downloads weights, accepts arbitrary checkpoints or executes shell commands.


In [ ]:
require_stage("assets")
STAGES["launch"] = False
if "WORKBENCH_PROCESS" in globals() and WORKBENCH_PROCESS.poll() is None:
    raise RuntimeError("This notebook already owns a running workbench. Reuse the viewer cell; do not launch a second model.")
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError("Port 8003 is already in use. Reuse its existing workbench or stop its known owning process yourself.")
launch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
LOG_PATH = REPOSITORY / "artifacts/vlkeb" / f"workbench-{launch_id}.log"
log_handle = LOG_PATH.open("w")
try:
    WORKBENCH_PROCESS = subprocess.Popen(
        [str(PYTHON), "-m", "ke_repro.workbench", "--upstream", str(UPSTREAM),
         "--assets", str(ASSETS), "--manifest", str(REPOSITORY / "configs/vlkeb-assets.json"),
         "--storage", str(STORAGE), "--host", "127.0.0.1", "--port", str(PORT)],
        cwd=REPOSITORY, stdout=log_handle, stderr=subprocess.STDOUT)
finally:
    log_handle.close()
STAGES["launch"] = True
print(json.dumps({"pid": WORKBENCH_PROCESS.pid, "log": str(LOG_PATH),
                  "notice": "Process started; readiness is established by the progress cell, not this message."}, indent=2))


In [ ]:
# Finite status inspection: one short request and the last 50 log lines.
# Rerun this cell if the asset check/model construction is still in progress.
require_stage("launch")
print("Server process return code:", WORKBENCH_PROCESS.poll())
if LOG_PATH.is_file():
    print("\n".join(LOG_PATH.read_text(errors="replace").splitlines()[-50:]))
try:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/status", timeout=3) as response:
        state = json.load(response)
    # Do not print or save the per-process CSRF token.
    print(json.dumps({key: state.get(key) for key in
                      ("phase", "error", "busy", "job", "active_edit_id", "active_method")}, indent=2))
except (OSError, ValueError) as exc:
    print("Server not ready for HTTP yet:", type(exc).__name__)
if WORKBENCH_PROCESS.poll() is not None:
    raise RuntimeError("The workbench stopped. Inspect the preserved log before retrying.")


## 5. Open the official Colab viewer

The helper below shows the port through Colab's notebook viewer, without a public sharing tunnel. It is the official [`serve_kernel_port_as_iframe` helper](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py). Loading can continue while the page is visible; inspect the progress cell if it reports a failure.

1. Choose a development image or upload your own JPEG/PNG/WebP (maximum 5 MiB and 20 megapixels).
2. Ask a question and generate the original answer.
3. Choose the edit method, enter a short intended correction and click **Preview correction**. This computes and saves a real update, tests it, and restores the original weights exactly; the candidate is not active.
4. Inspect the selected image's source/provenance, actual before/after answers, strict whole-answer target match, and fixed unrelated probe's answer change. Development fixtures come from the authors' training data; uploads retain their image hash. One probe does not establish general locality or factual correctness. Uploaded images use the same benchmark-neighbor probe.
5. Click **Approve and activate** if you choose to activate this saved candidate as a local research operator, after reviewing any warnings. Approval is not factual verification.
6. Ask a paraphrase and a neighboring-image question; inspect failures as well as successes. Expand **Patch history** for method, image hash, outputs, approval and exact parameter hashes. Download evidence and the parameter patch if required.
7. Click **Roll back active edit**. Exact weight restoration is checked; generate again to inspect restored answers.
8. Open **Evaluation** for the configured local report: base, prompt-only correction, updated image + text RAG, context-aligned MEND and adapted fine-tuning. RAG uses pretrained encoders, not a retriever we trained. The dashboard shows actual denominators, abstentions/errors, locality and measured query latency; the checksum-verified saved completed report is copied during checkout setup, without rerunning the experiment. Absent reports produce no numerical scores.

Only one edit can remain active. Changing the image or question clears stale displayed outputs. A new process loads original weights, not an old patch. Expected answers, source retrieval and teacher-forced scores must not be mislabeled as successful free generation. Live candidate previews are demonstrations, separate from the saved benchmark measurements.


In [ ]:
require_stage("launch")
if WORKBENCH_PROCESS.poll() is not None:
    raise RuntimeError("The server is not running. Inspect the progress cell.")
from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=1000)


## 6. Optionally save your local evidence

Colab runtime files are temporary. The next cell downloads a private ZIP containing only this workbench's history JSON, its launch log and recorded runtime inventory. It does **not** include uploaded image bytes or the large parameter patches. Download a patch from its history link if needed. This cell never mounts Drive, publishes a URL, or commits artifacts to Git.

History can contain your own questions and intended answers. Review it before sharing. Author-provided counterfactual targets and failed generated answers remain evidence, not verified real-world facts. Keep upstream published numbers explicitly cited and separate from local measurements.


In [ ]:
require_stage("launch")
bundle = Path("/content") / f"editguard-workbench-evidence-{launch_id}.zip"
if bundle.exists():
    raise FileExistsError("Evidence ZIP already exists; preserve it or choose a new filename.")
records = sorted(STORAGE.glob("*/record.json")) if STORAGE.exists() else []
with zipfile.ZipFile(bundle, "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for record in records:
        if record.resolve().is_relative_to(STORAGE.resolve()):
            archive.write(record, "history/" + record.relative_to(STORAGE).as_posix())
    for path in (LOG_PATH, inventory_path):
        if path.is_file():
            archive.write(path, "context/" + path.name)
    archive.writestr("provenance.json", json.dumps({
        "wrapper_commit": WRAPPER_COMMIT, "upstream_commit": UPSTREAM_COMMIT,
        "exported_at_utc": datetime.now(timezone.utc).isoformat(),
        "history_record_count": len(records), "includes_parameter_patches": False,
        "includes_uploaded_image_bytes": False,
        "notice": "Generated answers are unverified. Upstream published results are not our measurements."
    }, indent=2))
print("Local evidence ZIP:", bundle.name, "bytes:", bundle.stat().st_size)
from google.colab import files
files.download(str(bundle))
